<a href="https://colab.research.google.com/github/Karthikreddy1010/Electric-poles-and-wires-detection/blob/main/Electricpole_image_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Electric Utility Pole**

In [ ]:
!unzip "/content/street_view_route_images (1).zip"


Archive:  /content/street_view_route_images (1).zip
replace street_view_route_images/pano_37.948598873214365_-122.06410702267193_heading_0.5024686560859664.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace street_view_route_images/pano_37.948598873214365_-122.06410702267193_heading_120.50246865608597.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import os
from pathlib import Path
from typing import List, Tuple, Optional, Generator
import glob
from tqdm import tqdm

class ElectricPolePreprocessor:
    def __init__(self, target_size: Tuple[int, int] = (640, 640)):
        self.target_size = target_size
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")

    def load_images_from_folder(self, folder_path: str,
                              extensions: List[str] = None,
                              limit: int = None) -> List[Tuple[str, np.ndarray]]:
        """Load images from a folder with their file paths, with optional limit"""
        if extensions is None:
            extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff', '*.tif']

        image_paths = []
        for ext in extensions:
            image_paths.extend(glob.glob(os.path.join(folder_path, ext)))
            image_paths.extend(glob.glob(os.path.join(folder_path, ext.upper())))

        # Remove duplicates and sort
        image_paths = sorted(list(set(image_paths)))

        # Limit the number of images if specified
        if limit and limit > 0:
            image_paths = image_paths[:limit]

        images = []
        valid_paths = []

        print(f"Found {len(image_paths)} images in folder")

        for img_path in tqdm(image_paths, desc="Loading images"):
            try:
                img = cv2.imread(img_path)
                if img is not None:
                    images.append(img)
                    valid_paths.append(img_path)
                else:
                    print(f"Warning: Could not load {img_path}")
            except Exception as e:
                print(f"Error loading {img_path}: {e}")

        print(f"Successfully loaded {len(images)} images")
        return list(zip(valid_paths, images))

    def create_batches(self, data: List, batch_size: int) -> Generator:
        """Create batches from data"""
        for i in range(0, len(data), batch_size):
            yield data[i:i + batch_size]

    def bilateral_filter_batch(self, images: np.ndarray) -> np.ndarray:
        """Apply bilateral filter to batch of images"""
        batch_size = images.shape[0]
        filtered_batch = np.zeros_like(images)

        for i in range(batch_size):
            filtered_batch[i] = cv2.bilateralFilter(
                images[i], d=5, sigmaColor=75, sigmaSpace=75
            )
        return filtered_batch

    def hsv_enhancement_batch(self, images: np.ndarray) -> np.ndarray:
        """Enhance brightness and contrast in HSV space"""
        batch_size = images.shape[0]
        enhanced_batch = np.zeros_like(images)

        for i in range(batch_size):
            hsv = cv2.cvtColor(images[i], cv2.COLOR_BGR2HSV)
            h, s, v = cv2.split(hsv)

            # Enhance V channel
            v_enhanced = cv2.convertScaleAbs(v, alpha=1.15, beta=5)
            v_enhanced = np.clip(v_enhanced, 0, 255)

            hsv_enhanced = cv2.merge([h, s, v_enhanced])
            enhanced_batch[i] = cv2.cvtColor(hsv_enhanced, cv2.COLOR_HSV2BGR)

        return enhanced_batch

    def clahe_enhancement_batch(self, images: np.ndarray) -> np.ndarray:
        """Apply CLAHE to batch of images"""
        batch_size = images.shape[0]
        enhanced_batch = np.zeros_like(images)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

        for i in range(batch_size):
            lab = cv2.cvtColor(images[i], cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)

            l_enhanced = clahe.apply(l)
            lab_enhanced = cv2.merge([l_enhanced, a, b])
            enhanced_batch[i] = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2BGR)

        return enhanced_batch

    def gamma_correction_batch(self, images: np.ndarray, gamma: float = 1.2) -> np.ndarray:
        """Apply gamma correction to batch of images"""
        inv_gamma = 1.0 / gamma
        table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

        batch_size = images.shape[0]
        corrected_batch = np.zeros_like(images)

        for i in range(batch_size):
            corrected_batch[i] = cv2.LUT(images[i], table)

        return corrected_batch

    def edge_enhancement_batch(self, images: np.ndarray) -> np.ndarray:
        """Extract enhanced edges for batch of images"""
        batch_size = images.shape[0]
        edge_maps = np.zeros((batch_size, self.target_size[1], self.target_size[0]), dtype=np.uint8)

        for i in range(batch_size):
            gray = cv2.cvtColor(images[i], cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, threshold1=25, threshold2=100)

            kernel = np.ones((3, 3), np.uint8)
            edges_closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
            edge_maps[i] = edges_closed

        return edge_maps

    def resize_batch(self, images: np.ndarray) -> np.ndarray:
        """Resize batch of images to target size"""
        batch_size = images.shape[0]
        resized_batch = np.zeros((batch_size, self.target_size[1], self.target_size[0], 3), dtype=np.uint8)

        for i in range(batch_size):
            resized_batch[i] = cv2.resize(images[i], self.target_size)

        return resized_batch

    def save_preprocessing_steps(self, image: np.ndarray, original_path: str, output_folder: str):
        """Save each preprocessing step for a single image"""
        filename = Path(original_path).stem

        # Step 1: Resize
        resized = cv2.resize(image, self.target_size)
        resized_path = os.path.join(output_folder, f"{filename}_1_resized.jpg")
        cv2.imwrite(resized_path, resized)

        # Step 2: Bilateral filtering
        filtered = cv2.bilateralFilter(resized, d=5, sigmaColor=75, sigmaSpace=75)
        filtered_path = os.path.join(output_folder, f"{filename}_2_bilateral_filtered.jpg")
        cv2.imwrite(filtered_path, filtered)

        # Step 3: HSV enhancement
        hsv = cv2.cvtColor(filtered, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv)
        v_enhanced = cv2.convertScaleAbs(v, alpha=1.15, beta=5)
        v_enhanced = np.clip(v_enhanced, 0, 255)
        hsv_enhanced = cv2.merge([h, s, v_enhanced])
        hsv_result = cv2.cvtColor(hsv_enhanced, cv2.COLOR_HSV2BGR)
        hsv_path = os.path.join(output_folder, f"{filename}_3_hsv_enhanced.jpg")
        cv2.imwrite(hsv_path, hsv_result)

        # Step 4: CLAHE enhancement
        lab = cv2.cvtColor(hsv_result, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l_enhanced = clahe.apply(l)
        lab_enhanced = cv2.merge([l_enhanced, a, b])
        clahe_result = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2BGR)
        clahe_path = os.path.join(output_folder, f"{filename}_4_clahe_enhanced.jpg")
        cv2.imwrite(clahe_path, clahe_result)

        # Step 5: Gamma correction
        gamma = 1.2
        inv_gamma = 1.0 / gamma
        table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
        gamma_result = cv2.LUT(clahe_result, table)
        gamma_path = os.path.join(output_folder, f"{filename}_5_gamma_corrected.jpg")
        cv2.imwrite(gamma_path, gamma_result)

        # Step 6: Edge detection
        gray = cv2.cvtColor(gamma_result, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, threshold1=25, threshold2=100)
        kernel = np.ones((3, 3), np.uint8)
        edges_closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
        edge_path = os.path.join(output_folder, f"{filename}_6_edge_map.jpg")
        cv2.imwrite(edge_path, edges_closed)

        # Step 7: Final combined result (RGB + Edge overlay)
        overlay = gamma_result.copy()
        overlay[edges_closed > 0] = [0, 255, 0]  # Green edges
        final_path = os.path.join(output_folder, f"{filename}_7_final_with_edges.jpg")
        cv2.imwrite(final_path, overlay)

        print(f"Saved all preprocessing steps for {filename}")

    def process_folder(self,
                      input_folder: str,
                      batch_size: int = 8,
                      output_folder: Optional[str] = None,
                      limit: int = None) -> List[Tuple[str, np.ndarray]]:
        """
        Process all images in a folder in batches and save each preprocessing step

        Args:
            input_folder: Path to folder containing images
            batch_size: Number of images per batch
            output_folder: Folder to save processed images
            limit: Maximum number of images to process (None for all)

        Returns:
            List of tuples (file_path, final_processed_image)
        """
        # Load all images with optional limit
        image_data = self.load_images_from_folder(input_folder, limit=limit)
        if not image_data:
            print("No images found in folder!")
            return []

        results = []

        # Create batches
        batches = list(self.create_batches(image_data, batch_size))

        print(f"Processing {len(image_data)} images in {len(batches)} batches...")

        # Create output folder structure
        if output_folder:
            os.makedirs(output_folder, exist_ok=True)

            # Create subfolders for each step
            step_folders = [
                "01_resized",
                "02_bilateral_filtered",
                "03_hsv_enhanced",
                "04_clahe_enhanced",
                "05_gamma_corrected",
                "06_edge_maps",
                "07_final_with_edges"
            ]

            for folder in step_folders:
                os.makedirs(os.path.join(output_folder, folder), exist_ok=True)

        for batch_idx, batch in enumerate(tqdm(batches, desc="Processing batches")):
            file_paths, images = zip(*batch)

            # Convert list to numpy array for batch processing
            batch_np = np.stack(images)

            # Step 1: Resize
            batch_resized = self.resize_batch(batch_np)

            # Step 2: Bilateral filtering
            batch_filtered = self.bilateral_filter_batch(batch_resized)

            # Step 3: HSV enhancement
            batch_hsv = self.hsv_enhancement_batch(batch_filtered)

            # Step 4: CLAHE enhancement
            batch_clahe = self.clahe_enhancement_batch(batch_hsv)

            # Step 5: Gamma correction
            batch_gamma = self.gamma_correction_batch(batch_clahe, gamma=1.2)

            # Step 6: Edge enhancement
            edge_maps = self.edge_enhancement_batch(batch_gamma)

            # Store results and save images
            for i, file_path in enumerate(file_paths):
                filename = Path(file_path).stem

                # Get all processed versions
                resized_img = batch_resized[i]
                filtered_img = batch_filtered[i]
                hsv_img = batch_hsv[i]
                clahe_img = batch_clahe[i]
                gamma_img = batch_gamma[i]
                edge_img = edge_maps[i]

                # Create final overlay
                final_img = gamma_img.copy()
                final_img[edge_img > 0] = [0, 255, 0]  # Green edges

                results.append((file_path, final_img))

                # Save all steps if output folder specified
                if output_folder:
                    # Save individual steps
                    cv2.imwrite(os.path.join(output_folder, "01_resized", f"{filename}.jpg"), resized_img)
                    cv2.imwrite(os.path.join(output_folder, "02_bilateral_filtered", f"{filename}.jpg"), filtered_img)
                    cv2.imwrite(os.path.join(output_folder, "03_hsv_enhanced", f"{filename}.jpg"), hsv_img)
                    cv2.imwrite(os.path.join(output_folder, "04_clahe_enhanced", f"{filename}.jpg"), clahe_img)
                    cv2.imwrite(os.path.join(output_folder, "05_gamma_corrected", f"{filename}.jpg"), gamma_img)
                    cv2.imwrite(os.path.join(output_folder, "06_edge_maps", f"{filename}.jpg"), edge_img)
                    cv2.imwrite(os.path.join(output_folder, "07_final_with_edges", f"{filename}.jpg"), final_img)

        print(f"Successfully processed {len(results)} images")
        return results

    def visualize_preprocessing_steps(self, image: np.ndarray, save_path: Optional[str] = None):
        """Visualize each preprocessing step for a single image"""
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        axes = axes.ravel()

        # Original
        axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axes[0].set_title('1. Original Image', fontsize=12, fontweight='bold')
        axes[0].axis('off')

        # Resized
        resized = cv2.resize(image, self.target_size)
        axes[1].imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f'2. Resized to {self.target_size}', fontsize=12, fontweight='bold')
        axes[1].axis('off')

        # Bilateral filtered
        filtered = cv2.bilateralFilter(resized, d=5, sigmaColor=75, sigmaSpace=75)
        axes[2].imshow(cv2.cvtColor(filtered, cv2.COLOR_BGR2RGB))
        axes[2].set_title('3. Bilateral Filtered', fontsize=12, fontweight='bold')
        axes[2].axis('off')

        # HSV enhanced
        hsv = cv2.cvtColor(filtered, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv)
        v_enhanced = cv2.convertScaleAbs(v, alpha=1.15, beta=5)
        hsv_enhanced = cv2.merge([h, s, v_enhanced])
        hsv_result = cv2.cvtColor(hsv_enhanced, cv2.COLOR_HSV2BGR)
        axes[3].imshow(cv2.cvtColor(hsv_result, cv2.COLOR_BGR2RGB))
        axes[3].set_title('4. HSV Enhanced', fontsize=12, fontweight='bold')
        axes[3].axis('off')

        # CLAHE enhanced
        lab = cv2.cvtColor(hsv_result, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l_enhanced = clahe.apply(l)
        lab_enhanced = cv2.merge([l_enhanced, a, b])
        clahe_result = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2BGR)
        axes[4].imshow(cv2.cvtColor(clahe_result, cv2.COLOR_BGR2RGB))
        axes[4].set_title('5. CLAHE Enhanced', fontsize=12, fontweight='bold')
        axes[4].axis('off')

        # Gamma corrected
        gamma = 1.2
        inv_gamma = 1.0 / gamma
        table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
        gamma_result = cv2.LUT(clahe_result, table)
        axes[5].imshow(cv2.cvtColor(gamma_result, cv2.COLOR_BGR2RGB))
        axes[5].set_title('6. Gamma Corrected', fontsize=12, fontweight='bold')
        axes[5].axis('off')

        # Edge map
        gray = cv2.cvtColor(gamma_result, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 25, 100)
        kernel = np.ones((3, 3), np.uint8)
        edges_closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
        axes[6].imshow(edges_closed, cmap='gray')
        axes[6].set_title('7. Edge Map', fontsize=12, fontweight='bold')
        axes[6].axis('off')

        # Final with edge overlay
        final_img = gamma_result.copy()
        final_img[edges_closed > 0] = [0, 255, 0]  # Green edges
        axes[7].imshow(cv2.cvtColor(final_img, cv2.COLOR_BGR2RGB))
        axes[7].set_title('8. Final + Edge Overlay', fontsize=12, fontweight='bold')
        axes[7].axis('off')

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()


# Main execution
if __name__ == "__main__":
    # Initialize preprocessor
    preprocessor = ElectricPolePreprocessor(target_size=(640, 640))

    # Process first 2000 images
    results = preprocessor.process_folder(
        input_folder="/content/street_view_route_images",  # Your actual folder path
        batch_size=8,
        output_folder="processed_results",
        limit=2000  # This will process only the first 2000 images
    )

    print(f"\nProcessing complete! Processed {len(results)} images.")
    print(f"All preprocessing steps saved in 'processed_results' folder with subfolders for each step.")

Using device: cuda
Found 2000 images in folder


Loading images: 100%|██████████| 2000/2000 [00:07<00:00, 279.92it/s]


Successfully loaded 2000 images
Processing 2000 images in 250 batches...


Processing batches: 100%|██████████| 250/250 [01:42<00:00,  2.43it/s]

Successfully processed 2000 images

Processing complete! Processed 2000 images.
All preprocessing steps saved in 'processed_results' folder with subfolders for each step.
